In [10]:
from core.file_manager import preprocess_file_manager

from settings.main_settings import test_settings

import numpy as np

import SimpleITK as sitk


In [11]:
settings = test_settings().get_setting_dictionary()
preprocessed_steps = settings['preprocessed_steps']
preprocessing_steps_list = settings['preprocessing_steps_list']
channels = settings['channels']
original_data_folder = settings['original_data_folder']
target_spacing = settings['target_spacing']
crop_size = settings['crop_size']

file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)

In [12]:
from skimage.measure import marching_cubes
import plotly.graph_objects as go
import plotly.io as pio



image= file_manager.load_file('0_nifty','197')

mask = sitk.GetArrayFromImage(image['lesion'])

spacing = image['lesion'].GetSpacing()   # (x, y, z)

verts, faces, _, _ = marching_cubes(
    mask.astype(np.uint8),
    level=0.9,
    spacing=(spacing[2], spacing[1], spacing[0])  # array is (z,y,x)
)

fig = go.Figure(
    go.Mesh3d(
        x=verts[:, 2],
        y=verts[:, 1],
        z=verts[:, 0],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color="red",
        opacity=0.5,
        flatshading=True,
    )
)

fig.update_layout(
    # scene=dict(aspectmode="data"),
    width=900,
    height=900,
)

pio.renderers.default = "browser"
fig.show()

In [13]:
print(image["lesion"].GetSpacing())

(0.78125, 0.78125, 5.199999809265137)


In [14]:
from scipy import ndimage

labels, num = ndimage.label(mask)

print(f"Number of connected components: {num}")

sizes = ndimage.sum(mask, labels, index=range(1, num + 1))
print(sizes)

Number of connected components: 3
[ 81. 306.  63.]


In [15]:
from scipy.ndimage import label

labels, _ = label(mask)

for i in range(1, 4):
    print(f"Component {i}: {(labels == i).sum()} voxels")

Component 1: 81 voxels
Component 2: 306 voxels
Component 3: 63 voxels
